In [1]:
import xarray as xr
import numpy as np 

# Notebook dedicated to calculating the kinetic energy spectrum in the Fourier domain

In [2]:
file_path_to_dataset = '/Users/malenekarlsen/Documents/Master/Vår26/MachineLearning/Advanced_ML/train.nc'

In [3]:
ds = xr.open_dataset(file_path_to_dataset)

/var/folders/sq/5plwdlr95mscbwq45zg5whjc0000gn/T/ipykernel_81747/565364118.py:1: FutureWarning: In a future version, xarray will not decode the variable 'time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(file_path_to_dataset)


In [4]:
ds

<xarray.Dataset> Size: 3GB
Dimensions:              (run: 300, time: 86, lev: 2, y: 64, x: 64)
Coordinates:
  * time                 (time) timedelta64[ns] 688B 41 days 16:00:00.0000288...
  * lev                  (lev) int32 8B 1 2
  * y                    (y) float64 512B 7.812e+03 2.344e+04 ... 9.922e+05
  * x                    (x) float64 512B 7.812e+03 2.344e+04 ... 9.922e+05
Dimensions without coordinates: run
Data variables:
    psi                  (run, time, lev, y, x) float32 845MB ...
    q                    (run, time, lev, y, x) float32 845MB ...
    q_forcing_advection  (run, time, lev, y, x) float32 845MB ...
Attributes: (12/24)
    pyqg:L:          1000000.0
    pyqg:M:          65536
    pyqg:W:          1000000.0
    pyqg:beta:       1.5e-11
    pyqg:del2:       0.8
    pyqg:delta:      0.25
    ...              ...
    pyqg:tc:         86400
    pyqg:tmax:       311040000
    pyqg:twrite:     1000.0
    pyqg_params:     {'nx': 256, 'dt': 3600, 'tmax': 311040000, 'tavestart': ...
    reference:       https://pyqg.readthedocs.io/en/latest/index.html
    title:           pyqg: Python Quasigeostrophic Model

In [5]:
y = ds['y']
x = ds['x']
psi = ds['psi'].isel(time = slice(30, None))
Ny = len(y)
Nx = len(x)

# Hvordan finne hastighetsfeltet ved bruk av $\psi$

**Vi "vet" fra mekanikk:**

The stramfunction is a scalar function defined for an incompressible flow. The derivative of the streamfunction with respect to x yields the negative velocity component along the y-direction. The derivative of the streamfunction with respect to y yields the velocity component in the x-direction.

$$
u = \frac{\partial \psi}{\partial y} \quad \textbf{,} \quad v = - \frac{\partial \psi}{\partial x} 
$$

Source: Gjevik, B., & Fagerland, M. W. (2004). Feltteori og vektoranalyse: forelesninger og oppgaver i mek1100. Matematisk institutt, UiO.

But calculating this derivative for the whole grid with two layers for 43 days is a bit too much for the Macbook to handle, so instead we will take advantage of the Fourier Space. The Fourier transform of a derivative $f'(x)$ of a function is obtained by multiplication of its Fourier transform by : $ik$. 

Source: https://www.math.utah.edu/~gustafso/s2014/3150/pdeNotes/fourierTransform-PeterOlver2013.pdf

Also explained in the following book: 

$$
\psi (x,y,t) = \sum_{K} \tilde{\psi} (\bold{k} ,t) e^{ik \cdot x}
$$

Which is Equation 11.9 in Chapter 11: Vallis, G. K. (2017). Geostrophic Theory. In Atmospheric and Oceanic Fluid Dynamics: Fundamentals and Large-Scale Circulation (pp. 171–212). chapter, Cambridge: Cambridge University Press.

SO our way of calculating the velocity fields are done by multiplying the imaginary number $i$ and the spatial frequency $k$ with our fourier transform:

$$
\psi(\psi'(x,y,t)) = ik \hat{f}(k)
$$

In [6]:
import sympy as sp

In [7]:
#first - find the fourier transform of psi 
fft_two_dim = np.fft.fft2(psi) / (Nx * Ny) #normalising by Nx and Ny as in the article 

In [8]:
dx = float(ds['x'][1] - ds['x'][0])
dy = float(ds['y'][1] - ds['y'][0]) #diff between grid points

In [9]:
#calculate the spatial frequency 
kx = np.fft.fftfreq(Nx, d = dx) * 2 * np.pi
ky = np.fft.fftfreq(Ny, d = dy) * 2 * np.pi

In [10]:
i = 1j
u_fft = -i * ky[np.newaxis, np.newaxis, np.newaxis, :, np.newaxis] * (fft_two_dim) #None for all axes except from y
v_fft = i * kx[np.newaxis, np.newaxis, np.newaxis, np.newaxis, :] * (fft_two_dim) #None for all axes except from y


We are working with a two dimensional model so we are using: 

$$
KE = \frac{1}{2} \sum_{kx,ky}  |\hat{u}^2| |\hat{v}^2|
$$

But in cases where we assume isotropy, which means the turbulent flow is the same for all directions of the dataset. Then we can calculate a wavenumber $\mathcal{k}$, which is a total wavenumber. This simplification allows us to plot the energy spectrum against the single quantity of $\mathcal{k}$!

$$
\mathcal{k} = \frac{1}{2} (kx^2 + ky^2)^{\frac{1}{2}}
$$

Source is lecture notes from GEO4926

In [11]:
KE = (1/2) * ((np.abs(u_fft)**2) + (np.abs(v_fft)**2))
KE_mean = KE.mean(axis = (0,1,2))

In [12]:
KE_mean

array([[0.00000000e+00, 1.58635807e-08, 1.61584920e-07, ...,
        7.18260725e-07, 1.61584920e-07, 1.58635807e-08],
       [8.42557925e-08, 7.18498000e-08, 2.68774695e-07, ...,
        9.22473810e-07, 2.68161273e-07, 6.97577039e-08],
       [4.26615121e-07, 4.15585090e-07, 7.37521442e-07, ...,
        1.66385605e-06, 7.53398672e-07, 4.12222987e-07],
       ...,
       [1.29128875e-06, 1.34556032e-06, 1.86301869e-06, ...,
        2.96631885e-06, 1.89442198e-06, 1.32280375e-06],
       [4.26615121e-07, 4.12222987e-07, 7.53398672e-07, ...,
        1.60593548e-06, 7.37521442e-07, 4.15585090e-07],
       [8.42557925e-08, 6.97577039e-08, 2.68161273e-07, ...,
        9.19515281e-07, 2.68774695e-07, 7.18498000e-08]], shape=(64, 64))

In [13]:
kx2d, ky2d = np.meshgrid(kx, ky) #creating meshgrid for the wavelengths
K = np.sqrt(kx2d**2 + ky2d**2)

In [14]:
bins = np.linspace(0, K.max(), Nx//2)
KE_isotropy = np.zeros(len(bins)-1, dtype=(float))
k_centred = np.zeros(len(bins)-1, dtype=(float))

for t in range(len(bins)-1):
    mask = (K >= bins[t]) & (K < bins[t+1])
    if mask.sum() > 0:
        KE_isotropy[t] = KE_mean[mask].mean() 
        k_centred[t] = 0.5 * (bins[t] + bins[t+1])

In [ ]:
#Plotting function
import matplotlib.pyplot as plt
plt.figure(figsize = (8,6))
plt.loglog(k_centred, KE_isotropy, color = 'b')
plt.scatter(k_centred, KE_isotropy, color = 'm', alpha = 0.5)
plt.title(f'KE energy spectrum')
plt.xlabel(r'Wavenumber [$m^{-1}$]')
plt.ylabel(r'Kinetic Energy [$\frac{m^2}{s^2}$]')
plt.grid(True, alpha = 0.3)